In [33]:
import os
import docx2txt
import json
from together import Together

# Initialize Together AI client

# Initialize Together AI client
together_client = Together(api_key='46d9e04509efe0eb914936cc15d795f2104779b993b0bbc1b821fa5c49b20567')

def read_and_chunk_full_text(file_path, max_chars=60000):
    # Extract full text from the DOCX file
    full_text = docx2txt.process(file_path)
    full_text = full_text.replace('\n', ' ').strip()  # Replace newlines with spaces

    chunks = []
    text_length = len(full_text)
    start = 0
    chunk_id = 0

    while start < text_length:
        end = min(start + max_chars, text_length)
        # Ensure we don't split in the middle of a word
        if end < text_length and full_text[end] != ' ':
            end = full_text.rfind(' ', start, end)
            if end == -1 or end <= start:
                end = min(start + max_chars, text_length)
        chunk_text = full_text[start:end].strip()
        if chunk_text:
            chunk = {
                'id': chunk_id,
                'text': chunk_text,
            }
            chunks.append(chunk)
            chunk_id += 1
        start = end
    return chunks

def summarize_with_llama(text):
    # Use Together AI's LLaMA model for summarization
    text = str(text)
    prompt = f"Please extract the Arabic poetry excerpts and their analysis from the following text:\n\n{text}\n\n"
    print(f"Processing text chunk with prompt:\n{prompt}\n")

    messages = [
    {
        "role": "system",
        "content": """The following text is in Arabic, extracted from a docx file. 
        Your task is to extract specific information from the text. 
        The information you need to extract includes:
        - Poetry excerpts in Arabic.
        - Their corresponding analysis (if available).

        The output should be in the following dictionary format:
        {
            "poem": "Poetry Excerpt",
            "analysis": "Analysis"
        }

        The analysis could either be:
        - A direct explanation of the meaning of the poetry.
        - A single-word or multi-word translation in some cases.
        
        Important instructions:
        - Never attempt to provide your own analysis.
        - If no analysis is provided in the text, the analysis field should contain 'no_analysis.'
        - If the analysis is just a single-word translation, the analysis field should contain 'single_word.'
        - Do not add any additional information or notes beyond the specified analysis and poetry excerpts.
        
        Please follow these examples when extracting the information:

        Example 1:
        {
            "poem": "غيرَ أَنِّيِ وَدِدْتُ سَتْرَ صَديقٍ بَدلاً بِاسْتِفَادَةِ الأَنْبَاءِ",
            "analysis": "لكنني فضلت أن أستر صديقي على استفادة (معرفة) الأنباء عن عيوبه"
        }

        Example 2:
        {
            "poem": "وَلَوْ كُنْتُ أَدْرِي مَا الْحُبُّ مَا كُنْتُ أَحِبُّ الْحُبَّ",
            "analysis": "ولو كنت أعرف ما هو الحب، ما كنت أحببت الحب"
        }

        Example 3:
        {
            "poem": "قُلْنَ: هذا هَوىً، فَعَرِّجْ على الحقِّ - وخَلِّ الهَوى لِقَلْبٍ هَواءِ",
            "analysis": "قالت العيوب: هذا هوى (ضلال)، فعرج على الحق (تعال للحق)، واترك الضلال لقلب هواء (قلب ضعيف)."
        }

        Example 4 (Single-word translation):
        {
            "poem": "يا أخيِ: أينَ ريعُ ذاكَ اللِّقاءِ؟ أينَ ما كان بينَنا مِنْ صَفاءِ؟",
            "analysis": "single_word"
        }

        In this case, where the analysis is just a single-word translation, the analysis field should contain 'single_word.'

        Example 5 (No analysis provided):
        {
            "poem": "وَلَوْ كُنْتُ أَدْرِي مَا الْحُبُّ مَا كُنْتُ أَحِبُّ الْحُبَّ",
            "analysis": "no_analysis"
        }

        In cases where no analysis is provided in the text, the analysis field should contain 'no_analysis.'

        Additional Examples:
        
        Example 6:
        {
            "poem": "فإِخَالُ الذي تُدِيرُ على القَوْمِ حروباً دَوائرَ الأَرْحاءِ",
            "analysis": "يخيل إليَّ أن ما تديره على اللاعبين حروب دائرة الأرحاء (حجارة طواحينها تدور)"
        }

        Example 7:
        {
            "poem": "وأظنُّ افْتِراسَكَ القِرْنَ فالقِرْنَ مَنايا وَشيكةَ الإرْداءِ",
            "analysis": "ويهيأ إلي أن افتراسك القرن (الخصم) بعد الخصم منايا (ميتات) وشيكة الإرداء (سريعة الفتك)"
        }

        Example 8 (Multiple-word meanings):
        {
            "poem": "قالَ: يا بدرُ، أنتَ تَغْدِرُ بالسَّا 	رِيِ، وتُزْرِيِ بِزَوْرَةِ الحَسْناءِ",
            "analysis": "single_word"
        }

        In cases like Example 8, where the analysis consists of multiple-word meanings, the analysis field should contain 'single_word.'

        Do not provide any extra information or notes such as "Note: There are multiple poetry excerpts with analysis in the provided text." Only provide the poetry excerpts and their analysis as requested in the specified dictionary format."""
    },
    {
        "role": "user",
        "content": prompt
    }
]
    response = together_client.chat.completions.create(
        model="meta-llama/Meta-Llama-3-8B-Instruct-Lite",
        messages=messages,
        temperature=0.7,
        repetition_penalty=1.0,
        max_tokens=2000,
        stop=["<|eot_id|>", "<|eom_id|>"],
    )

    return response.choices[0].message.content

def process_chunks_with_llama(chunks):
    processed_results = []
    for chunk in chunks:
        try:
            result = summarize_with_llama(chunk['text'])
        except Exception as e:
            print(f"Error during processing chunk ID {chunk['id']}: {e}")
            result = ''  # Handle errors by returning empty result

        processed_chunk = {
            'id': chunk['id'],
            'original_text': chunk['text'],
            'processed_text': result
        }
        processed_results.append(processed_chunk)
    return processed_results

def build_json_output(processed_chunks):
    json_output = []
    for chunk in processed_chunks:
        chunk_data = {
            'id': chunk['id'],
            'original_text': chunk['original_text'],
            'processed_text': chunk['processed_text']
        }
        json_output.append(chunk_data)
    return json_output

def main():
    # Step 1: Read and chunk the DOCX file
    file_path = '/home/malek/AllamChallenge/data/raw/part-1.docx'  # Replace with your DOCX file path
    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        return

    max_chars = 2000  # Adjust based on the model's input limitations
    chunks = read_and_chunk_docx_file(file_path, max_chars)
    if not chunks:
        print("No text found in the DOCX file.")
        return

    # Step 2: Process chunks using Together AI's LLaMA model
    processed_chunks = process_chunks_with_llama(chunks)

    # Step 3: Build a JSON output from the API responses
    json_output = build_json_output(processed_chunks)

    # Step 4: Save the output to a JSON file
    with open('rag_output.json', 'w', encoding='utf-8') as f:
        json.dump(json_output, f, ensure_ascii=False, indent=4)
    print("RAG output saved to rag_output.json")

if __name__ == '__main__':
    main()


Processing text chunk with prompt:
Please extract the Arabic poetry excerpts and their analysis from the following text:

لقد نقل الشعر إلينا "الوجدان التاريخي" إن دق التعبير. اقرأ كتب المسعودي والطبري وابن الأثير وابن خلدون واليعقوبي في التاريخ وستعرف الكثير؛ ولكنك ستجد في الشعر معلومات أخرى غفلت عنها كتب التاريخ. ستجد فيه طريقة تفكير الناس في العصور المختلفة، وطريقة حياتهم، وتفاعلهم مع الأحداث. الشعر يُسِرُّ إليك بأمور غابت عن كتب التاريخ. ولا أظن أحداً من المؤرخين الذين ذكرتهم وصف لنا طريقة صنع الزلابية، ولا كيف يدحو الخباز رقاقته، ولا طريقة إعداد الساندويتش، ولا أن مترفي بغداد كانوا لا يجيزون لك أن تعض الخبزة بأسنانك ثم تغمسها في الطبق المشترك. كل هذا وصفه ابن الرومي، وكله موجود في مختاراتنا. لا أزعم لهذه السلسلة ما ليس لها. هي ليست أكثر من مختارات. وحتى في شعرائها فهي كذلك. فلن تضم كل الشعراء ولا نصفهم ولا عشرهم. ستضم فقط من أعتقدُ أنهم أبرزهم. فأنا أختار الشعر بذوقي؛ وبذوقي أختار الشعراء أيضاً. وأما التمثيل التاريخي فلا شأن لي به. قد أهمل قرناً أو قروناً ليس فيها شاعر عظيم. هذه س